In [ ]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

# --- 1. Load GUROBI Results (from our previous analysis) ---
gurobi_files = glob.glob(os.path.join('results', '**', '*_stats.txt'), recursive=True)
gurobi_data = []
for file_path in gurobi_files:
    data = {}
    with open(file_path, 'r') as f:
        for line in f:
            if ':' in line:
                key, value = line.split(':', 1)
                data[key.strip()] = value.strip()
    data['instance_base'] = data['Instance'].split('/')[-1].split('_lambda_')[0].replace('.csv', '')
    gurobi_data.append(data)

df_gurobi = pd.DataFrame(gurobi_data)
df_gurobi['disagreement_f1'] = pd.to_numeric(df_gurobi['Disagreement_Value_f1'])
df_gurobi['num_clusters_f2'] = pd.to_numeric(df_gurobi['Num_Clusters_Value_f2'])
print(f"Loaded {len(df_gurobi)} results from Gurobi.")


# --- 2. Load GENETIC ALGORITHM Results ---
ga_files = glob.glob(os.path.join('results_ga', 'pareto_*.csv'))
df_ga_all = pd.DataFrame()
for file_path in ga_files:
    instance_name = os.path.basename(file_path).replace('pareto_', '').replace('.csv', '')
    df_temp = pd.read_csv(file_path)
    df_temp['instance_base'] = instance_name
    df_ga_all = pd.concat([df_ga_all, df_temp])
print(f"Loaded {len(df_ga_all)} Pareto points from the Genetic Algorithm.")


# --- 3. Create Comparative Plots ---
unique_instances = sorted(df_gurobi['instance_base'].unique())
plots_dir = 'results_ga/comparative_plots'
os.makedirs(plots_dir, exist_ok=True)

for instance in unique_instances:
    df_g = df_gurobi[df_gurobi['instance_base'] == instance].drop_duplicates(subset=['num_clusters_f2'])
    df_a = df_ga_all[df_ga_all['instance_base'] == instance]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot Gurobi results (Exact Pareto Front)
    ax.plot(df_g['num_clusters_f2'], df_g['disagreement_f1'], 
            marker='s', markersize=10, linestyle='-', color='red', label='Gurobi (Fronteira Ótima)')

    # Plot GA results (Approximated Pareto Front)
    ax.plot(df_a['num_clusters_f2'], df_a['disagreement_f1'], 
            marker='o', markersize=8, linestyle='--', color='blue', label='Algoritmo Genético (Fronteira Aproximada)')

    ax.set_xlabel('Número de Clusters (k) - (Simplicidade)', fontsize=14)
    ax.set_ylabel('Desequilíbrio Esperado (f1) - (Qualidade)', fontsize=14)
    ax.set_title(f'Comparativo de Fronteiras de Pareto: {instance}', fontsize=16, pad=20)
    ax.legend(fontsize=12)
    ax.grid(True)
    
    # Save and show plot
    output_path = os.path.join(plots_dir, f"comp_{instance}.png")
    plt.savefig(output_path, dpi=300)
    print(f"Saved comparative plot for {instance} to {output_path}")
    plt.show()